# Competencia — Aprendizaje de Máquina 2026-10
## Parte 2: Clasificación de Textos Históricos con Deep Learning

Este notebook implementa la Parte 2 del proyecto de clasificación de textos históricos
en español/latín según su **década de origen** (39 clases, de 1500 a 1880). El objetivo
es superar el Private Score de referencia de la Parte 1 (0.29144) usando arquitecturas
de deep learning y transferencia de aprendizaje.

El enfoque adopta una estrategia en dos niveles diseñada para entrenamiento en CPU:

- **Nivel 1 — MLP sobre TF-IDF:** red neuronal densa que reemplaza el LinearSVC de la
  Parte 1 sobre la misma representación TF-IDF. Cumple el requisito de arquitectura de
  deep learning y establece un punto de comparación interno directo con la Parte 1.

- **Nivel 2 — Transfer Learning con MiniLM:** modelo principal. Se usa
  `paraphrase-multilingual-MiniLM-L12-v2`, un transformer multilingüe destilado de
  BERT entrenado en más de 50 idiomas (incluyendo español y latín). Genera embeddings
  densos de 384 dimensiones que capturan semántica contextual imposible de representar
  con TF-IDF. Sobre estos embeddings se entrena una cabeza clasificadora MLP con las
  41 features lingüísticas de la Parte 1 concatenadas. Esta arquitectura satisface los
  requisitos de deep learning, transferencia de aprendizaje y es ejecutable en CPU en
  2-4 horas sobre el dataset completo.

**Métrica objetivo:** Accuracy (leaderboard Kaggle) y F1-macro (optimización interna).

In [2]:
import pandas as pd
import numpy as np
import re, os, math, warnings, random
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from scipy.sparse import hstack, csr_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt

# ── Reproducibilidad ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Constantes del experimento ──
BASELINE_P1_LOCAL   = 0.2737   # F1-macro del mejor modelo Parte 1 (80/20 local)
BASELINE_P1_KAGGLE  = 0.29144  # Private Score Kaggle Parte 1 — score a superar
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

os.makedirs('./submissions', exist_ok=True)
os.makedirs('./model', exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'Device:  {DEVICE}')
print('✅ Listo')

PyTorch: 2.8.0
Device:  mps
✅ Listo


## 1. Carga de Datos

El dataset es idéntico al de la Parte 1:
- `train.csv`: 31.403 textos etiquetados con su década de origen
- `eval.csv`: 3.490 textos sin etiqueta sobre los cuales se generan las predicciones finales

La variable objetivo `decade` representa los tres primeros dígitos del año (ej: `164` para
la década de 1640), dando un total de **39 clases** desde 150 hasta 188.

In [3]:
df_train = pd.read_csv('./Data/train.csv')
df_eval  = pd.read_csv('./Data/eval.csv')

print(f'Train: {df_train.shape} | Eval: {df_eval.shape}')
print(f'Clases únicas: {df_train["decade"].nunique()}')
print(f'Distribución de clases (min/max textos): '
      f'{df_train["decade"].value_counts().min()} / '
      f'{df_train["decade"].value_counts().max()}')

Train: (31403, 2) | Eval: (3490, 2)
Clases únicas: 39
Distribución de clases (min/max textos): 754 / 848


El dataset contiene 31.403 textos de entrenamiento distribuidos en 39 clases con una
distribución casi uniforme — entre 754 y 848 textos por década. Esta uniformidad es
favorable para el entrenamiento: no hay sesgo por desbalance de clases y no se requiere
class weighting ni oversampling.

## 2. Preprocesamiento — Normalización OCR

Se reutiliza la función `normalize_ocr` de la Parte 1, que corrige artefactos tipográficos
y de digitalización sin tocar la ortografía arcaica del texto. El tokenizador de MiniLM
opera a nivel de subpalabra (WordPiece), por lo que formas como `hazer`, `dize` o `vna`
se tokenizarán en sus propios subwords — preservar esta señal es crítico para que el
modelo aprenda la evolución temporal del español.

Se eliminan además los 51 duplicados detectados en la Parte 1, quedando **31.352 textos**
para entrenamiento.

In [4]:
# ── Mapa de sustituciones OCR (idéntico a Parte 1) ──
CHAR_MAP = [
    ('\ufb01','fi'),('\ufb02','fl'),('\ufb00','ff'),('\ufb03','ffi'),('\ufb04','ffl'),
    ('\xe6','ae'),('\u0153','oe'),
    ('-\n',''),('- \n',''),('\xad',''),
    ('\xbb',' '),('\xab',' '),
    ('\u2018',"'"),("\u2019","'"),("\u201c",'"'),("\u201d",'"'),
    ('\xa3',' '),('\xa7',' '),('\xb6',' '),
    ('\u2020',' '),('\u2021',' '),('\u2022',' '),
    ('\u2014',' '),('\u2013',' '),
]

def normalize_ocr(text):
    text = str(text)
    for src, tgt in CHAR_MAP:
        text = text.replace(src, tgt)
    text = text.replace('\n',' ').replace('\r',' ').replace('\t',' ')
    return re.sub(r'  +', ' ', text).strip()

# ── Aplicar normalización y eliminar duplicados ──
data = df_train.drop_duplicates(subset='text').reset_index(drop=True).copy()
data['text_clean']    = data['text'].apply(normalize_ocr)
df_eval['text_clean'] = df_eval['text'].apply(normalize_ocr)

print(f'Train sin duplicados: {len(data)}')
print(f'Duplicados eliminados: {len(df_train) - len(data)}')

Train sin duplicados: 31352
Duplicados eliminados: 51


Se confirmaron los 51 duplicados detectados en la Parte 1. El dataset queda en 31.352
textos limpios. La normalización OCR preserva la ortografía arcaica intacta — las
sustituciones solo corrigen artefactos de digitalización (ligaduras, guiones de corte
de línea, símbolos tipográficos) que no aportan señal temporal.

## 3. Extracción de Features Lingüísticas Históricas

Se reutilizan las **41 features numéricas** de la Parte 1, que capturan características
morfológicas, ortográficas y tipográficas correlacionadas con la época del texto. Estas
features se concatenarán más adelante a los embeddings de MiniLM antes de la capa
clasificadora, aportando señal explícita que el transformer puede no capturar
directamente desde el texto crudo — en particular los patrones regex de latín y
ortografía arcaica.

In [5]:
# ── Patrones regex históricos (idénticos a Parte 1) ──
RE_LONG_S    = re.compile(r'[bcdfghjklmnpqrstvwxyz]f[aeiouáéíóú]', re.I)
RE_V_AS_U    = re.compile(r'\bvn[aeiouáéíóú]|\bvn\b|\bvm\b', re.I)
RE_ROMAN     = re.compile(
    r'\b(M{1,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})'
    r'|CM|CD|XC|XL|IX|IV|D?C{2,3}|L?X{2,3}|V?I{2,4})\b')
RE_LATIN_END = re.compile(r'\b\w{3,}(orum|ibus|atis|endi|antis|entis)\b', re.I)
RE_LATIN_NOM = re.compile(r'\b\w{3,}(um|us|ae)\b', re.I)
RE_QU_ARC    = re.compile(r'\bqu[aou]\w', re.I)
RE_SS        = re.compile(r'ss', re.I)
RE_FF        = re.compile(r'ff', re.I)
RE_CION      = re.compile(r'\b\w{3,}cion\b', re.I)
RE_TION      = re.compile(r'\b\w{3,}tion\b', re.I)
RE_SION      = re.compile(r'\b\w{3,}sion\b', re.I)
RE_ABBREV    = re.compile(r'\b[A-Za-z]{1,4}\.')
RE_MCASE     = re.compile(r'\b[A-Z][a-z]{1,}[A-Z]\w*\b')
RE_RDIAC     = re.compile(r'[àâãäāăąèêëēĕěîïīĭôõōŏùûüūŭ]', re.I)
RE_SEMI      = re.compile(r';')
RE_COLON     = re.compile(r':')
RE_PAREN     = re.compile(r'[()]')

def extract_features(text):
    words   = text.split()
    n       = max(len(words), 1)
    nc      = max(len(text), 1)
    lengths = [len(w) for w in words]
    counts  = Counter(text)
    total   = len(text) or 1
    entropy = -sum((c/total)*math.log2(c/total) for c in counts.values()) if text else 0
    vowels  = sum(1 for c in text.lower() if c in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    cons    = sum(1 for c in text.lower() if c.isalpha()
                  and c not in 'aeiouáéíóúàèìòùäëïöüâêîôû')
    sents   = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    ns      = max(len(sents), 1)
    return {
        'wl_mean':    np.mean(lengths) if lengths else 0,
        'wl_std':     np.std(lengths)  if lengths else 0,
        'wl_p75':     np.percentile(lengths, 75) if lengths else 0,
        'wl_p90':     np.percentile(lengths, 90) if lengths else 0,
        'ratio_long': sum(1 for l in lengths if l > 10) / n,
        'ratio_short':sum(1 for l in lengths if l <= 2) / n,
        'ratio_med':  sum(1 for l in lengths if 3 <= l <= 6) / n,
        'char_entropy':   entropy,
        'cv_ratio':       cons / vowels if vowels > 0 else 0,
        'word_char_ratio':n / nc,
        'comma_rate':     text.count(',') / n,
        'period_rate':    text.count('.') / n,
        'semicolon_rate': len(RE_SEMI.findall(text))  / n,
        'colon_rate':     len(RE_COLON.findall(text)) / n,
        'paren_rate':     len(RE_PAREN.findall(text)) / n,
        'excl_rate':      text.count('!') / n,
        'quest_rate':     text.count('?') / n,
        'total_punct':    sum(1 for c in text if c in '.,;:!?()[]{}') / n,
        'long_s_rate':    len(RE_LONG_S.findall(text))    / n,
        'v_as_u_rate':    len(RE_V_AS_U.findall(text))    / n,
        'latin_case':     len(RE_LATIN_END.findall(text)) / n,
        'latin_nom':      len(RE_LATIN_NOM.findall(text)) / n,
        'qu_archaic':     len(RE_QU_ARC.findall(text))    / n,
        'ss_rate':        len(RE_SS.findall(text))         / n,
        'ff_rate':        len(RE_FF.findall(text))         / n,
        'cion_rate':      len(RE_CION.findall(text))       / n,
        'tion_rate':      len(RE_TION.findall(text))       / n,
        'sion_rate':      len(RE_SION.findall(text))       / n,
        'abbrev_rate':    len(RE_ABBREV.findall(text))     / n,
        'roman_rate':     len(RE_ROMAN.findall(text))      / n,
        'rare_diac':      len(RE_RDIAC.findall(text))      / nc,
        'mixed_case':     len(RE_MCASE.findall(text))      / n,
        'ttr':         len(set(w.lower() for w in words)) / n,
        'upper_ratio': sum(1 for c in text if c.isupper()) / nc,
        'digit_ratio': sum(1 for c in text if c.isdigit()) / nc,
        'alpha_ratio': sum(1 for c in text if c.isalpha()) / nc,
        'all_caps':    sum(1 for w in words if w.isupper() and len(w)>1) / n,
        'pure_digit':  sum(1 for w in words if w.isdigit()) / n,
        'n_words':     float(n),
        'avg_sent_len':n / ns,
        'n_sentences': float(ns),
    }

print('Extrayendo features train...')
feats_train = pd.DataFrame(
    data['text_clean'].apply(extract_features).tolist()
).fillna(0)

print('Extrayendo features eval...')
feats_eval = pd.DataFrame(
    df_eval['text_clean'].apply(extract_features).tolist()
).fillna(0)

ALL_FEATS = feats_train.columns.tolist()
print(f'✅ Features extraídas: {len(ALL_FEATS)}')

Extrayendo features train...
Extrayendo features eval...
✅ Features extraídas: 41


Se extrajeron las 41 features lingüísticas históricas sobre los 31.352 textos de
entrenamiento y los 3.490 de evaluación. El conjunto incluye features de morfología
de palabras, estructura del texto, puntuación y patrones regex de latín y ortografía
arcaica — las mismas que en la Parte 1, donde demostraron aportar señal discriminativa
significativa sobre el TF-IDF solo.

## 4. Partición de Datos y Encoding de Etiquetas

Se realiza una partición estratificada 80/20 para evaluar los modelos localmente antes
de generar predicciones finales. El `LabelEncoder` convierte las 39 décadas (valores
enteros como 150, 151, ..., 188) a índices contiguos 0–38, formato requerido por
PyTorch para clasificación multiclase con `CrossEntropyLoss`.

In [6]:
# ── Encoding de etiquetas: décadas → índices 0-38 ──
le = LabelEncoder()
y_encoded = le.fit_transform(data['decade'].values)

print(f'Clases: {le.classes_[:5]} ... {le.classes_[-5:]}')
print(f'Índices: 0 ... {len(le.classes_) - 1}')

# ── Partición estratificada 80/20 ──
idx = np.arange(len(data))
idx_train, idx_val = train_test_split(
    idx, test_size=0.2, random_state=SEED, stratify=y_encoded
)

y_train = y_encoded[idx_train]
y_val   = y_encoded[idx_val]

print(f'\nTrain: {len(idx_train)} textos | Val: {len(idx_val)} textos')
print(f'Clases en train: {len(np.unique(y_train))} | '
      f'Clases en val: {len(np.unique(y_val))}')

Clases: [150 151 152 153 154] ... [184 185 186 187 188]
Índices: 0 ... 38

Train: 25081 textos | Val: 6271 textos
Clases en train: 39 | Clases en val: 39


La partición estratificada garantiza representación de las 39 décadas en train y
validación. El LabelEncoder mapea las décadas originales (150–188) a índices 0–38
— la transformación inversa `le.inverse_transform()` se usará al final para recuperar
las décadas reales en el archivo de submission.

## 5. Nivel 1 — MLP sobre TF-IDF

Antes de pasar al modelo principal de transfer learning, se entrena una red neuronal
densa (MLP) sobre la misma representación TF-IDF de la Parte 1. Esto cumple el
requisito de arquitectura de deep learning y establece un punto de comparación interno
directo: si el MLP supera el F1-macro de 0.2737 del LinearSVC de la Parte 1 sobre el
mismo split 80/20, confirma que la arquitectura neuronal agrega valor incluso sin
embeddings contextuales.

La arquitectura usa dos capas ocultas con ReLU y Dropout, entrada de dimensión igual
al número de features TF-IDF + 41 features numéricas, y salida de 39 clases.
Dado que la matriz TF-IDF es sparse, se convierte a tensor denso por batches dentro
del DataLoader para no materializar toda la matriz en memoria de una vez.